## Ingest ***circuits.csv*** file
1. Read the file using spark dataframe reader API
2. Add Metadata Columns 
     - Source File
     - Ingestion Timestamp
3. Write to bronze delta table    

#### Step 0 - Run configuration notebooks and set variables

In [0]:
dbutils.widgets.text("p_batch_id", "")

v_batch_id = dbutils.widgets.get("p_batch_id")
print(v_batch_id) 

In [0]:
%run ../00_common/01_configuration

In [0]:
%run ../00_common/02_bronze_functions

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema}.circuits"

#### Step 1 - Read the CSV file using the dataframe reader API

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

circuits_Schema = StructType([
                    StructField('circuitId'  , StringType()),
                    StructField('url'        , StringType()),
                    StructField('circuitName', StringType()),
                    StructField('lat'        , DoubleType()),
                    StructField('long'       , DoubleType()),
                    StructField('locality'   , StringType()),
                    StructField('country'    , StringType())
                ])

In [0]:
circuits_df = (
    spark.read
        .format('csv')
        .option('header', True)  # Used to skip the header row 
        #.option('inferSchema', True)  # Used to infer the data types. Possible issue --> goes through the data twice (read and infer the data types)
        .schema(circuits_Schema)  # Predefined schema
        .option('mode', 'failfast')  # Fail if there is any error reading the data (i.e. the readed data format is different from defined schema)
        .load(source_file)
)

In [0]:
# circuits_df.show()

In [0]:
# display(circuits_df)

#### Step 2 - Add Metadata Columns
- Source File
- Ingestion Timestamp

In [0]:
from pyspark.sql import functions as F
circuits_final_df = add_ingestion_metadata(circuits_df)

In [0]:
# display(circuits_final_df) 

#### Step 3 - Write to bronze delta table

In [0]:
# circuits_final_df = circuits_final_df.withColumn("batch_id", F.lit(v_batch_id)

# (
#     circuits_final_df
#         .write
#         .format('delta') # Recommended format
#         .mode('overwrite') # Could be append or overwrite
#         .partitionBy('batch_id')
#         .option('replaceWhere', f"batch_id = '{v_batch_id')
#         .saveAsTable(table_name)
# )

In [0]:
write_to_bronze (
    input_df = circuits_final_df,
    target_table = table_name,
    batch_id = v_batch_id
)

In [0]:
%sql
-- SELECT * FROM formula1_inc.bronze.circuits


In [0]:
# spark.table(table_name).display()